# Lecture 24 (Clustering)

## Exercise 24.1 (whiten)

Create your own implementation of [`scipy.cluster.vq.whiten`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.vq.whiten.html).

Assume we have _n_ data points, each being a _d_-dimensional vector _p_<sub>_i_</sub>, and let _p_<sub>_ij_</sub> denote
the _j_-th coordinate of the data point _i_. For each coordinate we compute the mean µ<sub>_j_</sub>  = (&sum;<sub>_i_=1.._n_</sub> _p_<sub>_ij_</sub>) / _n_, the variance σ<sub>_j_</sub><sup>2</sup> = (&sum;<sub>_i_=1.._n_</sub> (_p<sub>ij</sub>_ - µ<sub>_j_</sub>)<sup>2</sup>) / _n_, and the standard deviation σ<sub>_j_</sub> = (σ<sub>_j_</sub><sup>2</sup>)<sup>0.5</sup>. The method `whiten` replaces _p_<sub>_ij_</sub> by _p_<sub>_ij_</sub> / σ<sub>_j_</sub>.

You can either implement your method with plain Python using lists of tuples representing data-points or using `numpy`. For `numpy` be aware of the keyword arguments `axis` and `keepdims` for the `numpy.sum` method.

    > whiten([(1.9, 2.3, 1.7),
              (2.3, 1.8, 5.2),
              (1.5, 2.5, 2.2),
              (0.8, 0.6, 1.7)])
    [[3.4298319862895843, 3.1155131228015436, 1.16619037896906],
     [4.1519018781400225, 2.4382276613229474, 3.567170570964184],
     [2.7077620944391456, 3.386427307392982, 1.5091875492540778],
     [1.4441397837008778, 0.8127425537743157, 1.16619037896906]]

In [8]:
from math import sqrt
import numpy as np


def whiten(data):
    n = len(data)
    d = len(data[0])
    sum_ = [0] * d
    for obs in data:
        for i, value in enumerate(obs):
            sum_[i] += value

    mean = [x / n for x in sum_]

    var = [0] * d
    for obs in data:
        for i, value in enumerate(obs):
            var[i] += (value - mean[i]) ** 2

    return [[x / sqrt(v / n) for x, v in zip(obs, var)] for obs in data]


def np_whiten(data):
    n = len(data)
    mean = np.sum(data, axis=0, keepdims=True) / n
    var = np.sqrt(np.sum((data - mean) ** 2, axis=0, keepdims=True) / n)
    return data / var


def np_whiten2(data):
    return data / np.std(data, axis=0, keepdims=True)


data = [[1.9, 2.3, 1.7],
        [2.3, 1.8, 5.2],
        [1.5, 2.5, 2.2],
        [0.8, 0.6, 1.7]]

print(whiten(data))
print(np_whiten(data))
print(np_whiten2(data))

from scipy.cluster.vq import whiten

print(whiten(data))

[[3.4298319862895843, 3.1155131228015436, 1.16619037896906], [4.1519018781400225, 2.4382276613229474, 3.567170570964184], [2.7077620944391456, 3.386427307392982, 1.5091875492540778], [1.4441397837008778, 0.8127425537743157, 1.16619037896906]]
[[3.42983199 3.11551312 1.16619038]
 [4.15190188 2.43822766 3.56717057]
 [2.70776209 3.38642731 1.50918755]
 [1.44413978 0.81274255 1.16619038]]
[[3.42983199 3.11551312 1.16619038]
 [4.15190188 2.43822766 3.56717057]
 [2.70776209 3.38642731 1.50918755]
 [1.44413978 0.81274255 1.16619038]]
[[3.42983199 3.11551312 1.16619038]
 [4.15190188 2.43822766 3.56717057]
 [2.70776209 3.38642731 1.50918755]
 [1.44413978 0.81274255 1.16619038]]


## Exercise 24.2 (centroid for one cluster)

Prove that the centroid _c_ of a _d_-dimensional point set _C_ containing _n_ points is the average point, i.e. the
point _c_ = (&sum;<sub>_p_&in;_C_</sub> _p_) / _n_ minimizes the distortion &sum;<sub>_p_&in;_C_</sub> |_p_ - _c_|<sup>2</sup>, where |_p_ - _c_|<sup>2</sup> = &sum;<sub>_j_=1.._d_</sub> (_p<sub>j</sub>_ - _c<sub>j</sub>_)<sup>2</sup>.

_Note_. This is a plain math exercise.

_Hint_. Observe that it is sufficient to consider each coordinate independently and find the minimum point using differentiation.

In [2]:
'''

The problem is to find c = (c_x, c_y) minimizing

     sum_{p in C} |p - c| ** 2
   = sum_{p in C} ( (p_x - c_x) ** 2 + (p_y - c_y) ** 2 )
   = sum_{p in C} (p_x - c_x) ** 2 + sum_{p in C} (p_y - c_y) ** 2

i.e., C is a fixed constant point set and c_x and c_y are the variables.

corresponds to minimizing

   sum_{p in C} (p_x - c_x) ** 2
   sum_{p in C} (p_y - c_y) ** 2

independently for c_x and c_y, respectively.

These are each clearly 2nd degree polynomials in c_x and c_y, and are minimized where the derivative is zero.

The derivative for

   sum_{p in C} (p_x - c_x) ** 2

with respect to to c_x is

   sum_{p in C} -2 * (p_x - c_x)

that is zero when

       -2 * sum_{p in C} (p_x - c_x) = 0
   <=> sum_{p in C} p_x = |C| * c_x
   <=> c_x = (sum_{p in C} p_x) / |C|

'''

'\n\nThe problem is to find c = (c_x, c_y) minimizing\n\n     sum_{p in C} |p - c| ** 2\n   = sum_{p in C} ( (p_x - c_x) ** 2 + (p_y - c_y) ** 2 )\n   = sum_{p in C} (p_x - c_x) ** 2 + sum_{p in C} (p_y - c_y) ** 2\n\ni.e., C is a fixed constant point set and c_x and c_y are the variables.\n\ncorresponds to minimizing\n\n   sum_{p in C} (p_x - c_x) ** 2\n   sum_{p in C} (p_y - c_y) ** 2\n\nindependently for c_x and c_y, respectively.\n\nThese are each clearly 2nd degree polynomials in c_x and c_y, and are minimized where the derivative is zero.\n\nThe derivative for\n\n   sum_{p in C} (p_x - c_x) ** 2\n\nwith respect to to c_x is\n\n   sum_{p in C} -2 * (p_x - c_x)\n\nthat is zero when\n\n       -2 * sum_{p in C} (p_x - c_x) = 0\n   <=> sum_{p in C} p_x = |C| * c_x\n   <=> c_x = (sum_{p in C} p_x) / |C|\n\n'

## Exercise 24.3* (k-means clustering in 1D)

Assume we have a sorted list _L_ of _N_ ≥ 1 real values _x_<sub>1</sub> ≤ ··· ≤ _x<sub>N</sub>_,
and we want to solve the k-means problem for this input _optimally_, i.e. to find _K_
centroids _c_<sub>1</sub>,...,_c<sub>K</sub>_, 1 ≤ _K_ ≤ _N_, such that the distortion
&sum;<sub>_i_=1.._N_</sub> min<sub>_j_=1.._K_</sub> (_x<sub>i</sub>_ - _c<sub>j</sub>_)<sup>2</sup> is minimized.

Let _D_(_k_, _n_) denote the minimum distortion for _k_ centroids for _x_<sub>1</sub>, ...,_x<sub>n</sub>_,
where 1 ≤ _k_ ≤ _K_ and 1 ≤ _n_ ≤ _N_.

_D_(_k_, _n_) can be computed by the following recurrence:

* _D_(1, _n_) = &sum;<sub>_i_=1.._n_</sub> (_x<sub>i</sub>_ - µ)<sup>2</sup>,
  where µ = (&sum;<sub>_i_=1.._n_</sub> _x<sub>i</sub>_) / _n_

* _D_(_k_, _n_) = min<sub>_i_=1.._n_-1</sub> (_D_(_k_ - 1, _i_) + &sum;<sub>_j_=_i_+1.._n_</sub>
  (_x<sub>j</sub>_ - (&sum;<sub>_t_=_i_+1.._n_</sub> _x<sub>t</sub>_) / (_n_ - _i_))<sup>2</sup>)

Create a function `k_mean_1D(K, L)` that finds `K` optimal centroids for the list `L`. The function should return a pair containing the total distortion and the list of centroids.

_Hint_. Use dynamic programming.

In [ ]:
from functools import cache as memoize

# The below solutions generate solutions that are linked lists
# represted as a tuple(recursive tuple for (c_1,...,c_{k-1}), c_k),
# i.e. a new solution can be constructed using O(1) space + space
# for a previous solution, that can be shared among several extended
# solutions.

#################################################################
#  Memoization solution O(k * n^3)


def k_mean_1D(k, L):
    def one_cluster(i, j):
        ''' consider L[i:j] to be one cluster '''

        centroid = sum(L[i:j]) / (j - i)
        cost = sum((x - centroid)**2 for x in L[i:j])
        return cost, centroid

    @memoize
    def solve(k, n):
        ''' find k optimal centroids for L[0:n] '''

        cost, solution = one_cluster(0, n)

        if k > 1:
            for i in range(1, n-1):
                cost_head, solution_head = solve(k - 1, i)
                cost_tail, solution_tail = one_cluster(i, n)

                cost_ = cost_tail + cost_head
                if cost_ < cost:
                    cost = cost_
                    solution = (solution_head, solution_tail)

        return cost, solution

    L = sorted(L)
    return solve(k, len(L))


#################################################################
#  2 * Memoization solution O(n^3 + k * n^2)


def k_mean_1D_double_memoize(k, L):
    @memoize
    def one_cluster(i, j):
        ''' consider L[i:j] to be one cluster '''

        centroid = sum(L[i:j]) / (j - i)
        cost = sum((x - centroid)**2 for x in L[i:j])
        return cost, centroid

    @memoize
    def solve(k, n):
        ''' find k optimal centroids for L[0:n] '''

        cost, solution = one_cluster(0, n)

        if k > 1:
            for i in range(1, n-1):
                cost_head, solution_head = solve(k - 1, i)
                cost_tail, solution_tail = one_cluster(i, n)

                cost_ = cost_tail + cost_head
                if cost_ < cost:
                    cost = cost_
                    solution = (solution_head, solution_tail)

        return cost, solution

    L = sorted(L)
    return solve(k, len(L))


#################################################################
#  Tabulation solution O(k * n^3)


def k_mean_1D_table(K, L):
    def one_cluster(i, j):
        centroid = sum(L[i:j]) / (j - i)
        cost = sum((x - centroid)**2 for x in L[i:j])
        return cost, centroid

    L = sorted(L)
    N = len(L)
    solved = {(1, n): one_cluster(0, n) for n in range(1, N + 1)}

    for k in range(2, K + 1):
        for n in range(1, N + 1):
            cost, solution = one_cluster(0, n)
            for i in range(1, n - 1):
                cost_head, solution_head = solved[(k - 1, i)]
                cost_tail, solution_tail = one_cluster(i, n)

                cost_ = cost_tail + cost_head
                if cost_ < cost:
                    cost = cost_
                    solution = (solution_head, solution_tail)
            solved[(k, n)] = (cost, solution)
    return solved[(K, N)]


#################################################################
#  Tabulation solution (only storing two rows) O(k * n^3)


def k_mean_1D_two_rows(K, L):
    def one_cluster(i, j):
        centroid = sum(L[i:j]) / (j - i)
        cost = sum((x - centroid)**2 for x in L[i:j])
        return cost, centroid

    L = sorted(L)
    N = len(L)

    previous = [one_cluster(0, n) if n else None for n in range(N + 1)]
    current = [None] * (N + 1)

    for k in range(2, K + 1):
        for n in range(1, N + 1):
            cost, solution = one_cluster(0, n)
            for i in range(1, n - 1):
                cost_head, solution_head = previous[i]
                cost_tail, solution_tail = one_cluster(i, n)
                cost_ = cost_tail + cost_head
                if cost_ < cost:
                    cost = cost_
                    solution = (solution_head, solution_tail)
            current[n] = (cost, solution)
        previous, current = current, previous

    return previous[N]


#################################################################
#  Tabulation solution (only storing two rows + prefix sums) O(k * n^2)


def prefix_sums(L):
    S = [0]
    for e in L:
        S.append(S[-1] + e)
    return S


def k_mean_1D_prefix_sum(K, L):
    def one_cluster(i, j):
        sum_x = sums[j] - sums[i]
        sum_xx = squares[j] - squares[i]
        centroid = sum_x / (j - i)
        cost = centroid * (centroid * (j - i ) - 2 * sum_x) + sum_xx
        return cost, centroid

    L = sorted(L)
    N = len(L)
    sums = prefix_sums(L)
    squares = prefix_sums(x**2 for x in L)

    previous = [one_cluster(0, n) if n else None for n in range(N + 1)]
    current = [None] * (N + 1)

    for k in range(2, K + 1):
        for n in range(1, N + 1):
            cost, solution = one_cluster(0, n)
            for i in range(1, n - 1):
                cost_head, solution_head = previous[i]
                cost_tail, solution_tail = one_cluster(i, n)
                cost_ = cost_tail + cost_head
                if cost_ < cost:
                    cost = cost_
                    solution = (solution_head, solution_tail)
            current[n] = (cost, solution)
        previous, current = current, previous

    return previous[N]


#################################################################
#  Tabulation solution (two rows + prefix sums + monotonic search) O(k * n)
#  See https://arxiv.org/abs/1701.07204


def k_mean_1D_monotonic(K, L):
    def one_cluster(i, j):
        sum_x = sums[j] - sums[i]
        sum_xx = squares[j] - squares[i]
        centroid = sum_x / (j - i)
        cost = centroid * (centroid * (j - i ) - 2 * sum_x) + sum_xx
        return cost, centroid

    L = sorted(L)
    N = len(L)
    sums = prefix_sums(L)
    squares = prefix_sums(x**2 for x in L)

    row = [one_cluster(0, n) if n else None for n in range(N + 1)]

    for k in range(2, K + 1):
        idx = N
        for n in range(N, 0, -1):
            cost, solution = row[n]
            for i in range(min(idx, n - 1), 0, -1):  # continue search where last search ended
                cost_head, solution_head = row[i]
                cost_tail, solution_tail = one_cluster(i, n)
                cost_ = cost_tail + cost_head
                if cost_ >= cost:  # early escape
                    break
                cost, solution = cost_, (solution_head, solution_tail)
                idx = i
            row[n] = (cost, solution)
    return row[N]


#################################################################
#  Main test

k_mean_1D_memoization = k_mean_1D

algorithms = [
    k_mean_1D_memoization,
    k_mean_1D_double_memoize,
    k_mean_1D_table,
    k_mean_1D_two_rows,
    k_mean_1D_prefix_sum,
    k_mean_1D_monotonic
]

k = 30

for f in algorithms:
    print(f.__name__, f(3, [1,2,7,5,6,3,11,12]))

import matplotlib.pyplot as plt
import time
from random import random

for f in algorithms:
    data = []
    for i in range(10,30):
        N = int(1.2 ** i)
        print(f.__name__, i, N)
        start = time.time()
        answer = f(k, [N * random() for _ in range(N)])
        end = time.time()
        data.append((N, end - start))
    plt.plot(*zip(*data), '.-', label=f.__name__, alpha=0.5)

plt.legend()
plt.yscale('log')
plt.xscale('log')
plt.xlabel('n')
plt.ylabel('time (seconds)')
plt.title('k-means for k = %s' % k)
plt.show()